# 01 — Exploratory Data Analysis

A guided tour of the bakery's operational data. Run cells top-to-bottom.

**Connects to**: the same Postgres database the backend uses.
**Refresh data**: re-run the loaders at the top after recording new sales/expenses in the app.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from db import load_sales, load_sale_items, load_expenses, load_production, load_stock

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 50)

sales = load_sales()
items = load_sale_items()
expenses = load_expenses()
production = load_production()
stock = load_stock()

print(f'Sales: {len(sales):,} receipts spanning {sales.occurred_at.min():%Y-%m-%d} → {sales.occurred_at.max():%Y-%m-%d}')
print(f'Sale items: {len(items):,}')
print(f'Expenses: {len(expenses):,}')
print(f'Production runs: {len(production):,}')
print(f'Stock rows: {len(stock):,}')

## 1. Daily revenue and profit

In [ ]:
daily = (
    sales.assign(day=sales['occurred_at'].dt.tz_localize(None).dt.date)
         .groupby('day')
         .agg(revenue=('total', 'sum'), profit=('profit', 'sum'), receipts=('id', 'count'))
         .reset_index()
)
daily['day'] = pd.to_datetime(daily['day'])
daily['rolling7_revenue'] = daily['revenue'].rolling(7, min_periods=1).mean()

fig = px.line(daily, x='day', y=['revenue', 'rolling7_revenue', 'profit'],
              title='Daily revenue, 7-day moving average, and profit',
              labels={'value': 'Amount (RWF)', 'day': 'Date', 'variable': 'Metric'})
fig.show()

## 2. Busy hours & day-of-week

In [ ]:
sales['hour'] = sales['occurred_at'].dt.tz_localize(None).dt.hour
sales['weekday'] = sales['occurred_at'].dt.tz_localize(None).dt.day_name()

weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heat = (sales.groupby(['weekday','hour']).size().reset_index(name='receipts')
             .pivot(index='weekday', columns='hour', values='receipts')
             .reindex(weekday_order))

plt.figure(figsize=(14, 5))
sns.heatmap(heat, annot=False, cmap='OrRd', linewidths=0.4,
            cbar_kws={'label': 'Receipts'})
plt.title('Receipts by weekday × hour-of-day')
plt.xlabel('Hour'); plt.ylabel('')
plt.show()

## 3. Top products by revenue and units

In [ ]:
top = (items.groupby(['product_name', 'category'])
             .agg(units=('quantity', 'sum'), revenue=('line_total', 'sum'),
                  cost=('unit_cost', lambda s: (s * items.loc[s.index, 'quantity']).sum()))
             .assign(profit=lambda d: d['revenue'] - d['cost'])
             .sort_values('revenue', ascending=False)
             .head(15))
top

In [ ]:
fig = px.bar(top.reset_index(), x='revenue', y='product_name', color='category',
             orientation='h', title='Top 15 products by revenue', height=500)
fig.update_yaxes(autorange='reversed')
fig.show()

## 4. Profitability mix by category

In [ ]:
by_cat = (items.groupby('category')
                .apply(lambda d: pd.Series({
                    'units': d['quantity'].sum(),
                    'revenue': d['line_total'].sum(),
                    'cost': (d['unit_cost'] * d['quantity']).sum(),
                }), include_groups=False)
                .assign(profit=lambda d: d['revenue'] - d['cost'],
                        margin_pct=lambda d: (d['revenue'] - d['cost']) / d['revenue'] * 100)
                .sort_values('revenue', ascending=False))
by_cat

## 5. Expenses vs revenue — monthly P&L summary

In [ ]:
monthly_rev = (sales.assign(month=sales['occurred_at'].dt.tz_localize(None).dt.to_period('M'))
                    .groupby('month')['total'].sum().rename('revenue'))
monthly_cogs = (sales.assign(month=sales['occurred_at'].dt.tz_localize(None).dt.to_period('M'))
                     .groupby('month')['cost_of_goods'].sum().rename('cogs'))
monthly_exp = (expenses.assign(month=expenses['incurred_on'].dt.to_period('M'))
                       .groupby('month')['amount'].sum().rename('expenses'))
pnl = pd.concat([monthly_rev, monthly_cogs, monthly_exp], axis=1).fillna(0)
pnl['gross_profit'] = pnl['revenue'] - pnl['cogs']
pnl['net_profit'] = pnl['gross_profit'] - pnl['expenses']
pnl

## 6. Stock health

In [ ]:
stock['headroom'] = stock['quantity'].astype(float) - stock['reorder_threshold'].astype(float)
low = stock[stock['headroom'] <= 0].sort_values('headroom')
print(f'{len(low)} of {len(stock)} items at/below their reorder threshold')
low.head(15)